# Trabalho Prático 1 - Aprendizagem Automática
## Previsão de Preços de Carros Usados (Kaggle Competition)

**Licenciatura em Engenharia de Sistemas e Tecnologias Informáticas** 
**Unidade Curricular:** Aprendizagem Automática  
**Ano Letivo:** 2025/2026 

---

### 1. Introdução e Objetivos
Este *notebook* documenta o processo de desenvolvimento de modelos de Aprendizagem Automática para prever o preço de carros usados, no âmbito da competição Kaggle da disciplina.

A metodologia adotada segue o pipeline clássico de Data Science:
1.  **Análise e Processamento de Dados (Feature Engineering):** Limpeza, normalização e criação de novas variáveis (*ratios*, categorias) para capturar padrões complexos.
2.  **Seleção de Modelos:** Teste de algoritmos de diferentes famílias (Lineares, Árvores, Boosting e Redes Neuronais) conforme exigido no enunciado.
3.  **Otimização (Grid Search & CV):** Ajuste fino de hiperparâmetros com validação cruzada para garantir a robustez e evitar *overfitting*.
4.  **Ensemble:** Combinação dos melhores modelos num *Voting Regressor* para maximizar a precisão final.

### 2. Identificação do Grupo
* **Aluno 1:** Bernardo Freitas (79295)
* **Aluno 2:** Afonso Figueiredo (79309)

In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os, joblib, re, random

# Scikit-Learn - Modelação e Processamento
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, r2_score

# Algoritmos
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor  # Variante de RF (Gradient Boosting)

# Configurações Globais
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Reprodutibilidade (Sementes)
RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print('Bibliotecas carregadas e reprodutibilidade configurada.')

### 3. Configuração de Execução

Para permitir um desenvolvimento ágil sem comprometer a qualidade final, definimos dois modos de execução:

* **`MODE = 'quick'`**: Utiliza menos *folds* na validação cruzada e uma grelha de hiperparâmetros reduzida. Ideal para testes de código e *debugging*.
* **`MODE = 'full'`**: Executa uma busca exaustiva (*Randomized/Grid Search*) com 5 *folds*. Este modo deve ser ativado para a submissão final.

Além disso, utilizamos `TARGET_LOG = True` para aplicar uma transformação logarítmica (`log1p`) à variável alvo (Preço). Isto é crucial pois a distribuição de preços tende a ser assimétrica (cauda longa), e os modelos convergem melhor com dados normalizados.

In [ ]:
# Seleção do Modo de Execução
MODE = 'quick'  # Alterar para 'full' na entrega final

if MODE == 'quick':
    CV_FOLDS = 3
    N_ITER_SEARCH = 10
    print("MODO RÁPIDO: Treino acelerado para testes.")
else:
    CV_FOLDS = 5 
    N_ITER_SEARCH = 30 
    print("MODO COMPLETO: Busca exaustiva para melhor score.")

TARGET_LOG = True 
TRAIN_N_JOBS = -1 # Usa todos os núcleos do CPU

print(f'Config: CV={CV_FOLDS}, Log-Target={TARGET_LOG}')

### 4. Carregamento dos Dados
Os ficheiros `train.csv` e `test.csv` são carregados para dataframes pandas. É feita uma verificação preliminar das dimensões para garantir que os dados foram lidos corretamente.

In [ ]:
try:
    df_train = pd.read_csv('../data/train.csv')
    df_test = pd.read_csv('../data/test.csv')
    sample_sub = pd.read_csv('../data/sample_submission.csv')
    print(f"Dados carregados com sucesso.\nTreino: {df_train.shape} | Teste: {df_test.shape}")
except FileNotFoundError:
    print("ERRO: Ficheiros não encontrados. Verifique a pasta '../data/'.")

### 5. Feature Engineering (Engenharia de Atributos)

Esta etapa é crítica para o desempenho do modelo. Transformamos os dados brutos em variáveis mais informativas:

1.  **Limpeza de Texto:** Extração de valores numéricos das colunas `engine` (HP, Litros) e `mileage`, removendo unidades e caracteres especiais.
2.  **Variáveis Temporais:** Cálculo da idade do veículo (`age`) com base no ano do modelo.
3.  **Variáveis Derivadas (Ratios):**
    * `hp_per_liter`: Indica a eficiência/performance do motor (carros desportivos tendem a ter valores mais altos).
    * `mileage_per_year`: Identifica uso intensivo do veículo.
4.  **Categorização e Flags:**
    * Criação de *flags* binárias para marcas de luxo (`is_luxury`) e transmissão automática.
    * Agrupamento de marcas/modelos raros em "Other" para reduzir a dimensionalidade do *One-Hot Encoding*.
    * Discretização da potência (`hp_category`) e idade (`age_category`) para capturar relações não-lineares.

In [ ]:
current_year = 2025

def extract_number(s, pattern):
    """Função auxiliar para extrair números via Regex."""
    if pd.isna(s): return np.nan
    m = re.search(pattern, str(s), flags=re.I)
    return float(m.group(1)) if m else np.nan

def feature_engineering(data, train_stats=None):
    df = data.copy()
    
    # 1. Normalização de Nomes
    if 'mileage' in df.columns: df.rename(columns={'mileage': 'milage'}, inplace=True)

    # 2. Extração Numérica
    df['hp'] = df['engine'].apply(lambda x: extract_number(x, r"(\d+\.?\d*)\s*(?:HP|bhp|PS)"))
    df['displ_l'] = df['engine'].apply(lambda x: extract_number(x, r"(\d+\.?\d*)\s*[lL]"))
    
    if df['milage'].dtype == 'O':
        df['milage'] = df['milage'].astype(str).str.replace(',', '').str.extract(r"(\d+\.?\d*)")[0].astype(float)

    # 3. Features Temporais
    df['age'] = current_year - pd.to_numeric(df['model_year'], errors='coerce')
    
    # 4. Flags Binárias
    for col in ['accident', 'clean_title']:
        df[f"{col}_flag"] = df.get(col, pd.Series(0)).astype(str).str.lower().isin(['yes', 'true', '1', 'y', 'sim']).astype(int)
    
    df['is_automatic'] = df['transmission'].astype(str).str.lower().str.contains('auto', na=False).astype(int)
    luxury_brands = ['BMW', 'Mercedes-Benz', 'Audi', 'Lexus', 'Porsche', 'Land Rover', 'Jaguar', 'Ferrari', 'Lamborghini']
    df['is_luxury'] = df['brand'].isin(luxury_brands).astype(int)

    # 5. Ratios (Tratamento de divisão por zero)
    _hp = df['hp'].fillna(df['hp'].median())
    _age = df['age'].replace(0, 1).fillna(5)
    _displ = df['displ_l'].replace(0, np.nan).fillna(df['displ_l'].median())
    
    df['hp_per_liter'] = _hp / _displ
    df['milage_per_year'] = df['milage'] / _age
    
    # 6. Binning (Categorização)
    df['age_category'] = pd.cut(df['age'], bins=[-1, 3, 7, 15, 100], labels=['novo', 'usado', 'velho', 'antigo'])
    df['hp_category'] = pd.cut(df['hp'].fillna(0), bins=[0, 150, 300, 500, 10000], labels=['basico', 'potente', 'desportivo', 'supercarro'])

    return df

# Aplicar FE
df_train_fe = feature_engineering(df_train)
df_test_fe = feature_engineering(df_test)

# 7. Frequency Encoding & Grouping (Baseado apenas no Treino para evitar Data Leakage)
for col in ['brand', 'model']:
    # Calcular frequências no treino
    freqs = df_train_fe[col].value_counts(normalize=True)
    # Manter apenas top 25, o resto vira 'Other'
    top_cats = freqs.nlargest(25).index
    
    df_train_fe[f'{col}_grp'] = df_train_fe[col].where(df_train_fe[col].isin(top_cats), 'Other')
    df_test_fe[f'{col}_grp'] = df_test_fe[col].where(df_test_fe[col].isin(top_cats), 'Other')

print("Feature Engineering concluída.")

### 6. Pipeline de Pré-processamento

Definimos um `ColumnTransformer` para tratar automaticamente os dados antes de entrarem nos modelos:
* **Variáveis Numéricas:** Imputação pela mediana (robusta a *outliers*) e normalização com `StandardScaler`.
* **Variáveis Categóricas:** Imputação pelo valor mais frequente e codificação *One-Hot* (`handle_unknown='ignore'` para lidar com categorias novas no teste).

In [ ]:
# Seleção de Features Finais
features = [
    'age', 'milage', 'hp', 'displ_l', 'hp_per_liter', 'milage_per_year', # Numéricas
    'brand_grp', 'model_grp', 'fuel_type', 'transmission', # Categóricas
    'accident_flag', 'clean_title_flag', 'is_luxury', 'is_automatic', # Binárias
    'age_category', 'hp_category' # Bins
]

X = df_train_fe[features].copy()
y = df_train_fe['price'].copy()

# Log-Transformation do Target (se ativado)
if TARGET_LOG:
    y = np.log1p(y)

# Identificar tipos
numeric_feats = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_feats = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Construção do Pipeline
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()) 
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_feats),
        ('cat', cat_transformer, categorical_feats)
    ],
    remainder='drop'
)

# Divisão Treino/Validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Dataset pronto. Features: {len(features)}")

### 7. Definição dos Modelos e Hiperparâmetros

De acordo com o enunciado, testamos algoritmos variados para encontrar o melhor compromisso entre viés e variância:

1.  **Ridge Regression (Linear):** Modelo base simples, com regularização L2 para evitar *overfitting*.
2.  **Random Forest (Bagging):** Robusto e capaz de capturar relações não-lineares.
3.  **XGBoost (Boosting):** Variante de árvore avançada (Gradient Boosting), geralmente o estado da arte para dados tabulares.
4.  **MLP Regressor (Rede Neuronal):** Modelo capaz de aprender interações complexas, usando a função de ativação ReLU.

Utilizamos `RandomizedSearchCV` (no modo completo) para explorar eficientemente o espaço de hiperparâmetros.

In [ ]:
models = []

def get_grid(small, large):
    return small if MODE == 'quick' else large

# 1. Ridge Regression
models.append(('Ridge', 
               Pipeline([('pre', preprocessor), ('model', Ridge())]),
               {'model__alpha': [0.1, 1.0, 10.0, 100.0]}))

# 2. Random Forest
rf_params = {
    'model__n_estimators': get_grid([50], [100, 300]),
    'model__max_depth': get_grid([10], [15, 25, None]),
    'model__min_samples_leaf': [1, 4]
}
models.append(('RF', 
               Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=TRAIN_N_JOBS))]),
               rf_params))

# 3. XGBoost
xgb_params = {
    'model__n_estimators': get_grid([100], [500, 1000]),
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 5, 7],
    'model__subsample': [0.7, 0.9]
}
models.append(('XGB', 
               Pipeline([('pre', preprocessor), ('model', XGBRegressor(random_state=RANDOM_STATE, n_jobs=TRAIN_N_JOBS, objective='reg:squarederror'))]),
               xgb_params))

# 4. MLP (Rede Neuronal)
mlp_params = {
    'model__hidden_layer_sizes': get_grid([(50,)], [(100, 50), (200, 100)]), 
    'model__alpha': [0.0001, 0.001],
    'model__max_iter': [500],
    'model__early_stopping': [True]
}
models.append(('MLP', 
               Pipeline([('pre', preprocessor), ('model', MLPRegressor(random_state=RANDOM_STATE))]),
               mlp_params))

print(f"Modelos configurados: {[m[0] for m in models]}")

### 8. Treino e Avaliação

Executamos o ciclo de treino para cada modelo. Os resultados são avaliados usando a métrica **RMSE (Root Mean Squared Error)**. Como transformámos o target com Log, revertemos a transformação (`expm1`) para calcular o erro real em Dólares ($).

In [ ]:
results_data = []
os.makedirs('../models', exist_ok=True)

# Função de avaliação auxiliar
def evaluate(model, X_v, y_v, name):
    preds = model.predict(X_v)
    # Reverter Log se necessário
    y_true = np.expm1(y_v) if TARGET_LOG else y_v
    y_pred = np.expm1(preds) if TARGET_LOG else preds
    y_pred = np.clip(y_pred, 0, None) # Garantir não-negativos
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'Model': name, 'RMSE': rmse, 'R2': r2}

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print("=== INÍCIO DO TREINO ===")
for name, pipe, params in models:
    print(f"\nTreinando {name}...")
    
    # Escolha entre Grid e Randomized Search
    if MODE == 'quick':
        search = GridSearchCV(pipe, params, cv=cv, scoring='neg_mean_squared_error', n_jobs=TRAIN_N_JOBS, verbose=3)
    else:
        search = RandomizedSearchCV(pipe, params, n_iter=N_ITER_SEARCH, cv=cv, 
                                    scoring='neg_mean_squared_error', n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE)
    
    try:
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        
        # Avaliar
        metrics = evaluate(best_model, X_val, y_val, name)
        metrics['Best Params'] = search.best_params_
        results_data.append(metrics)
        
        # Guardar modelo
        joblib.dump(best_model, f'../models/{name}_best.joblib')
        print(f" -> {name} RMSE: ${metrics['RMSE']:,.2f}")
        
    except Exception as e:
        print(f" -> Erro no {name}: {e}")

# Exibir Resumo
results_df = pd.DataFrame(results_data).sort_values('RMSE')
display(results_df)

### 9. Ensemble (Voting Regressor)

Para tentar superar a performance dos modelos individuais, criamos um **Voting Regressor**. Este meta-modelo faz a média das previsões dos melhores modelos treinados anteriormente (Ridge, RF, XGB, MLP). 

* **Porquê?** Modelos diferentes cometem erros em exemplos diferentes. Ao combiná-los, a variância do erro tende a diminuir, resultando numa previsão mais estável e generalista.

In [ ]:
estimators_list = []
model_names = ['XGB', 'RF', 'MLP', 'Ridge'] # Prioridade de inclusão

print("\nConfigurando Ensemble...")
for name in model_names:
    try:
        # Carrega o pipeline completo salvo
        loaded_pipe = joblib.load(f'../models/{name}_best.joblib')
        # Extrai apenas o modelo final (step 'model') para inserir no Voting
        estimators_list.append((name.lower(), loaded_pipe.named_steps['model']))
        print(f" -> {name} adicionado.")
    except FileNotFoundError:
        print(f" -> {name} ignorado (não encontrado).")

if len(estimators_list) >= 2:
    # O Ensemble precisa do seu próprio pipeline de pré-processamento
    ensemble_pipe = Pipeline([
        ('pre', preprocessor),
        ('voting', VotingRegressor(estimators=estimators_list, n_jobs=TRAIN_N_JOBS))
    ])
    
    # Treinar Ensemble
    ensemble_pipe.fit(X_train, y_train)
    
    # Avaliar
    ens_metrics = evaluate(ensemble_pipe, X_val, y_val, 'Ensemble')
    joblib.dump(ensemble_pipe, '../models/Ensemble_best.joblib')
    
    print(f"\nEnsemble Treinado. RMSE: ${ens_metrics['RMSE']:,.2f}")
    
    # Adicionar aos resultados
    results_df = pd.concat([results_df, pd.DataFrame([ens_metrics])], ignore_index=True).sort_values('RMSE')
    display(results_df)
else:
    print("Não há modelos suficientes para criar um Ensemble.")

### 10. Visualização Comparativa
Gráfico comparativo do desempenho (RMSE) de todos os modelos testados.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x='Model', y='RMSE', palette='viridis')
plt.title('Comparação de RMSE (Menor é Melhor)')
plt.ylabel('RMSE ($)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 11. Geração da Submissão (Kaggle)

Selecionamos automaticamente o modelo com o menor RMSE no conjunto de validação para gerar as previsões finais no conjunto de teste (`test.csv`).

In [ ]:
# 1. Identificar melhor modelo
best_row = results_df.iloc[0]
best_name = best_row['Model']
print(f"Melhor Modelo selecionado: {best_name}")

# 2. Carregar modelo
final_model = joblib.load(f'../models/{best_name}_best.joblib')

# 3. Preparar dados de teste (aplicar as mesmas transformações)
X_test = df_test_fe[features].copy()

# 4. Prever
preds_log = final_model.predict(X_test)

# 5. Reverter Log e tratar valores
preds_final = np.expm1(preds_log) if TARGET_LOG else preds_log
preds_final = np.clip(preds_final, 0, None)

# 6. Criar ficheiro
submission = pd.DataFrame({
    'id': df_test['id'],
    'price': preds_final
})

submission.to_csv('submission.csv', index=False)
print("Ficheiro 'submission.csv' gerado com sucesso!")
display(submission.head())